In [44]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import brier_score_loss, make_scorer, log_loss
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import shap

In [45]:
df = pd.read_csv("../Data/cleanedHealthData.csv")

In [46]:
## Establish weights for target variable
healthy_weight = df['target'].value_counts(normalize=True)['healthy']
diseased_weight = df['target'].value_counts(normalize=True)['diseased']

In [47]:
## Separate all the columns
numerical_cols = df.select_dtypes(include=['number']).columns
# numerical_cols = [col for col in numerical_cols if col != 'target']

categorical_cols = list(df.select_dtypes(exclude=['number']).columns)
categorical_cols = [col for col in categorical_cols if col != 'target']


In [86]:
## Set target as 1|0

df['target'] = df['target'].replace({'healthy': 0, 'diseased': 1})

y = df['target']
X = df.drop(columns='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [51]:
## Setup Numerical and Categorical Cols Transformers (KNNImputer, Scaler and OHE)

num_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3)), ## Reduce to 3 to save computation
    ('scaler', StandardScaler(with_mean=False)),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe', OneHotEncoder(drop = 'first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

## For Random Forest only 
imp_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3))
])

rf_preprocesser = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ] 
)

## Baseline Logistic Regression Model

In [ ]:
## Logistic Regression pipeline (0.210771)

base_models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
}

results = []
for model, pipeline in tqdm(base_models.items(), desc="Training Models"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

Training Models: 100%|██████████| 1/1 [03:12<00:00, 192.24s/it]


,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210771,0.612582


In [ ]:
def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = []

    if 'num' in preprocessor.named_transformers_:
        feature_names.extend(numeric_cols)

    if 'cat' in preprocessor.named_transformers_:
        ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
        ohe_names = ohe.get_feature_names_out(categorical_cols)
        feature_names.extend(ohe_names)

    return feature_names

## Use LASSO, Random Forest, KMeans for Feature Selection

In [ ]:
## LASSO Feature Selection
fs_models = {
    'Lasso': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(penalty='l1', solver='liblinear', random_state=42))
    ])
}

# results = []
for model, pipeline in tqdm(fs_models.items(), desc="Lasso Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    ## Get feature names (OHE produced more)
    pre = pipeline.named_steps['preprocessor']
    feature_names = get_feature_names(pre, numerical_cols, categorical_cols)
    
    lasso_importance = pd.Series(np.abs(pipeline['model'].coef_).flatten(), index=feature_names)
    lasso_features = set(lasso_importance[lasso_importance > 0].index)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df


Lasso Feature Selection: 100%|██████████| 1/1 [03:12<00:00, 192.40s/it]


,Model,Brier Score Loss,Log Loss
1,Lasso,0.210761,0.612558
0,LogisticRegression,0.210771,0.612582


In [ ]:
print(f"Lasso Feature Selection:\n{lasso_features}")

Lasso Feature Selection:
{'cholesterol', 'sugar_intake', 'pet_owner_Yes', 'meals_per_day', 'occupation_Driver', 'alcohol_consumption_Occasionally', 'job_type_Unemployed', 'occupation_Farmer', 'sunlight_exposure_Low', 'family_history_Yes', 'income', 'smoking_level_Light', 'water_intake', 'education_level_Master', 'occupation_Engineer', 'exercise_type_Mixed', 'bmi_corrected', 'blood_pressure', 'physical_activity', 'smoking_level_Non-smoker', 'exercise_type_Missing', 'alcohol_consumption_Regularly', 'height', 'insulin', 'daily_steps', 'caffeine_intake_Moderate', 'screen_time', 'sleep_quality_Poor', 'education_level_High School', 'job_type_Labor', 'exercise_type_Strength', 'sleep_quality_Good', 'device_usage_Moderate', 'age', 'job_type_Office', 'insurance_Yes', 'diet_type_Vegan', 'glucose', 'waist_size', 'sleep_hours', 'job_type_Tech', 'daily_supplement_dosage', 'stress_level', 'diet_type_Vegetarian', 'caffeine_intake_Missing', 'device_usage_Low', 'gender_Male', 'work_hours', 'weight', 'me

In [ ]:
## Random Forest Feature Selection
fs_models = {
    'RandomForest': Pipeline([
        ('preprocessor', rf_preprocesser),  ## No StandardScaler for RF
        ('model', RandomForestClassifier(random_state=42))
    ])
}

results = []
for model, pipeline in tqdm(fs_models.items(), desc="Random Forest Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    ## Get feature names (OHE produced more)
    pre = pipeline.named_steps['preprocessor']
    feature_names = get_feature_names(pre, numerical_cols, categorical_cols)
    
    rf_importance = pd.Series(pipeline['model'].feature_importances_, index=feature_names)
    rf_features = set(rf_importance[rf_importance > rf_importance.mean()].index)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

Random Forest Feature Selection: 100%|██████████| 1/1 [04:03<00:00, 243.77s/it]


,Model,Brier Score Loss,Log Loss
0,RandomForest,0.21366,0.619371


In [87]:
rf_importance.sort_values(ascending=False)

insulin                     0.038739
heart_rate                  0.038703
waist_size                  0.038585
sugar_intake                0.038533
daily_steps                 0.038463
                              ...   
occupation_Farmer           0.004096
occupation_Doctor           0.004080
occupation_Engineer         0.003997
job_type_Office             0.003881
gene_marker_flag_Missing    0.003460
Length: 63, dtype: float64

In [ ]:
print(f"Random Forest Feature Selection:\n{rf_features}")

Random Forest Feature Selection:
{'cholesterol', 'sugar_intake', 'income', 'water_intake', 'bmi_corrected', 'blood_pressure', 'physical_activity', 'height', 'insulin', 'daily_steps', 'screen_time', 'age', 'glucose', 'waist_size', 'sleep_hours', 'daily_supplement_dosage', 'stress_level', 'work_hours', 'weight', 'calorie_intake', 'heart_rate', 'mental_health_score'}


In [ ]:
## KNN uses permutation importance (which requires testing features repeatedly to check if loss metric changes)



In [ ]:
for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [ ]:
## XGB Feature Selection
fs_models = {
    'XGB': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))   ## No scaling needed
    ])
}

results = []
for model, pipeline in tqdm(fs_models.items(), desc="XGB Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    explainer = shap.TreeExplainer(pipeline['model'])
    shap_values = explainer.shap_values(X_train)
   
    
    shap_importance = pd.Series(np.abs(shap_values).mean(0), index=X.columns)
    xgb_feature_names = set(shap_importance.nlargest(30).index)

    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

XGB Feature Selection: 100%|██████████| 1/1 [00:06<00:00,  6.14s/it]


,Model,Brier Score Loss,Log Loss
0,XGB,0.218409,0.632276


In [ ]:
print(f"XGB Feature Selection:\n{xgb_feature_names}")

XGB Feature Selection:
{'cholesterol', 'sleep_quality', 'education_level', 'sugar_intake', 'diet_type', 'occupation', 'income', 'water_intake', 'bmi_corrected', 'blood_pressure', 'physical_activity', 'healthcare_access', 'height', 'insulin', 'daily_steps', 'screen_time', 'exercise_type', 'age', 'glucose', 'waist_size', 'sleep_hours', 'job_type', 'daily_supplement_dosage', 'stress_level', 'device_usage', 'work_hours', 'weight', 'calorie_intake', 'heart_rate', 'mental_health_score'}


In [88]:
## Filtered out 22 features (length of rf_features)

selected_features = (
    lasso_features & rf_features |
    lasso_features & xgb_feature_names |
    rf_features & xgb_feature_names
)


print(len(selected_features))
print("\n Final Selected Features:\n", selected_features)

22

 Final Selected Features:
 {'cholesterol', 'sugar_intake', 'income', 'water_intake', 'bmi_corrected', 'blood_pressure', 'physical_activity', 'height', 'insulin', 'daily_steps', 'screen_time', 'age', 'glucose', 'waist_size', 'sleep_hours', 'daily_supplement_dosage', 'stress_level', 'work_hours', 'weight', 'calorie_intake', 'heart_rate', 'mental_health_score'}


## Baseline Pipeline of all Models on Full Data

In [ ]:
models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', rf_preprocesser),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'XGBoost': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))
    ])
}

In [ ]:
results = []

for model, pipeline in tqdm(models.items(), desc="Training Baseline Models Full Data"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

baseline_results_df = pd.DataFrame(results).sort_values(by='Log Loss')
# results_df.to_csv('../Data/preliminaryResults.csv', index=False)
# results_df

Training Baseline Models Full Data: 100%|██████████| 6/6 [17:06<00:00, 171.14s/it]


In [ ]:
baseline_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210771,0.612582
2,GradientBoosting,0.210801,0.612651
1,RandomForest,0.213660,0.619371
5,XGBoost,0.218409,0.632276
4,KNN,0.251964,2.432118
3,DecisionTree,0.433200,15.614111


## Use Logistic Regression, Random Forest, Boosting Models to Evaluate Selection

In [56]:
## Filter dataset for selected features
X_train_selected = X_train[list(selected_features)]
X_test_selected = X_test[list(selected_features)]

selected_categorical_cols = [feature for feature in categorical_cols if feature in selected_features]
selected_numerical_cols = [feature for feature in numerical_cols if feature in selected_features]

## Adjust Preprocessor and Pipeline based on limited features

sel_preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, selected_numerical_cols),
        ('cat', cat_transformer, selected_categorical_cols)
    ]
)

sel_rf_preprocesser = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, selected_numerical_cols),
        ('cat', cat_transformer, selected_categorical_cols)
    ],
    remainder='drop' 
)

sel_models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', sel_rf_preprocesser),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'XGBoost': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))
    ])
}


In [ ]:
results = []

for model, pipeline in tqdm(sel_models.items(), desc="Training Baseline Models Selected Data"):
    pipeline.fit(X_train_selected, y_train)
    y_pred_prob = pipeline.predict_proba(X_test_selected)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

selected_results_df = pd.DataFrame(results).sort_values(by='Log Loss')
# results_df.to_csv('../Data/preliminaryResults.csv', index=False)
# results_df

Training Baseline Models Selected Data: 100%|██████████| 6/6 [16:10<00:00, 161.79s/it]


In [ ]:
baseline_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210771,0.612582
2,GradientBoosting,0.210801,0.612651
1,RandomForest,0.213660,0.619371
5,XGBoost,0.218409,0.632276
4,KNN,0.251964,2.432118
3,DecisionTree,0.433200,15.614111


In [ ]:
selected_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210607,0.612188
2,GradientBoosting,0.210727,0.612485
1,RandomForest,0.213666,0.619433
5,XGBoost,0.218696,0.632911
4,KNN,0.251318,2.494181
3,DecisionTree,0.424700,15.307740


## Hyperparameter Tuning

In [ ]:
## Set parameters for each model

param_grids = {
    "GradientBoosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [3, 4, 5],
        "model__subsample": [0.6, 0.8, 1.0],
    },
    "LogisticRegression": {
        "model__C": np.logspace(-3, 3, 10),
        "model__solver": ["liblinear"],
        "model__penalty": ["l2", "l1"],
    },
    "RandomForest": {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [None, 5, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    },
    "DecisionTree": {
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__criterion": ["gini", "entropy"],
    },
    'XGBoost': {
        "model__n_estimators": [200, 400, 800],
        "model__learning_rate": [0.001, 0.01, 0.1, 1],
        "model__max_depth": [2, 3, 4, 5, 6],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0]
    }
    
}

In [59]:
log_loss_scorer = make_scorer(log_loss, response_method='predict_proba', greater_is_better=False, labels=[0,1])
brier_scorer = make_scorer(brier_score_loss, response_method='predict_proba', greater_is_better=False, labels=[0,1])

scoring = {
    "log_loss": log_loss_scorer,
    "brier": brier_scorer
}

In [ ]:
X_train_selected.columns

Index(['cholesterol', 'sugar_intake', 'income', 'water_intake',
       'bmi_corrected', 'blood_pressure', 'physical_activity', 'height',
       'insulin', 'daily_steps', 'screen_time', 'age', 'glucose', 'waist_size',
       'sleep_hours', 'daily_supplement_dosage', 'stress_level', 'work_hours',
       'weight', 'calorie_intake', 'heart_rate', 'mental_health_score'],
      dtype='object')

In [85]:
## Features that were not selected
features_notUsed = list(set(X.columns).difference(set(X_train_selected.columns)))
features_notUsed

['sleep_quality',
 'smoking_level',
 'education_level',
 'alcohol_consumption',
 'meals_per_day',
 'diet_type',
 'occupation',
 'mental_health_support',
 'healthcare_access',
 'gene_marker_flag',
 'caffeine_intake',
 'sunlight_exposure',
 'exercise_type',
 'pet_owner',
 'insurance',
 'job_type',
 'device_usage',
 'gender',
 'family_history']

In [ ]:
## Perform Randomized Search on selected features
best_models = {}

## Need to use selected features models pipeline
for name, pipeline in sel_models.items():
    print(f"\n Running RandomizedSearchCV for {name}...")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grids[name],
        n_iter=10,              ## try 10 random combinations
        scoring=scoring,
        refit='log_loss', 
        cv=3,                   ## reduce to increase computation speed
        verbose=2,
        random_state=42,
        error_score='raise',
        n_jobs=4,               ## use 4 CPU cores
        pre_dispatch="2*n_jobs"
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    
    print(f"Best params for {name}: {search.best_params_}")
    print(f"Best CV log loss: {-search.best_score_:.4f}")

# Evaluate best models on test set
print("\n Test Log Loss:")
models_dict = {}
for name, model in best_models.items():
    y_pred_proba = model.predict_proba(X_test)
    ll = log_loss(y_test, y_pred_proba)
    models_dict[name]= ll

res_df = pd.DataFrame.from_dict(models_dict, orient='index').reset_index()
res_df.columns = ['Model', 'Test Log Loss']
res_df.to_csv("../Data/tunedFSAllModels.csv", index=False)


 Running RandomizedSearchCV for LogisticRegression...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for LogisticRegression: {'model__solver': 'liblinear', 'model__penalty': 'l1', 'model__C': np.float64(0.021544346900318832)}
Best CV log loss: 0.6097

 Running RandomizedSearchCV for RandomForest...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for RandomForest: {'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_depth': 5}
Best CV log loss: 0.6097

 Running RandomizedSearchCV for GradientBoosting...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for GradientBoosting: {'model__subsample': 0.6, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01}
Best CV log loss: 0.6096

 Running RandomizedSearchCV for DecisionTree...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for DecisionTree: {'model__min_samples_spl

AttributeError: 'dict' object has no attribute 'to_csv'

In [ ]:
models_dict = {}
for name, model in best_models.items():
    y_pred_proba = model.predict_proba(X_test)
    ll = log_loss(y_test, y_pred_proba)
    models_dict[name]= ll

res_df = pd.DataFrame.from_dict(models_dict, orient='index').reset_index()
res_df.columns = ['Model', 'Test Log Loss']
res_df = res_df.sort_values(by='Test Log Loss', ascending=True)
res_df.to_csv("../Data/tunedFSAllModels.csv", index=False)

In [ ]:
## Perform Randomized Search on all features
best_models = {}

for name, pipeline in models.items():
    print(f"\n Running RandomizedSearchCV for {name}...")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grids[name],
        n_iter=10,              ## try 10 random combinations
        scoring=scoring,
        refit='log_loss',  
        cv=3,                   ## reduce to increase computation speed
        verbose=2,
        random_state=42,
        error_score='raise',
        n_jobs=4,               ## use 4 CPU cores
        pre_dispatch="2*n_jobs"              
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    
    print(f"Best params for {name}: {search.best_params_}")
    print(f"Best CV log loss: {-search.best_score_:.4f}")

models_dict = {}
# Evaluate best models on test set
print("\n Test Log Loss:")
for name, model in best_models.items():
    y_pred_proba = model.predict_proba(X_test)
    ll = log_loss(y_test, y_pred_proba)
    models_dict[name]= ll
    print(f"{name}: {ll:.4f}")

res_df = pd.DataFrame.from_dict(models_dict, orient='index').reset_index()
res_df.columns = ['Model', 'Test Log Loss']
res_df = res_df.sort_values(by='Test Log Loss', ascending=True)
res_df.to_csv("../Data/tunedFullAllModels.csv", index=False)


 Running RandomizedSearchCV for LogisticRegression...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for LogisticRegression: {'model__solver': 'liblinear', 'model__penalty': 'l1', 'model__C': np.float64(0.004641588833612777)}
Best CV log loss: 0.6096

 Running RandomizedSearchCV for RandomForest...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for RandomForest: {'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_depth': 5}
Best CV log loss: 0.6096

 Running RandomizedSearchCV for GradientBoosting...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for GradientBoosting: {'model__subsample': 0.6, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01}
Best CV log loss: 0.6096

 Running RandomizedSearchCV for DecisionTree...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for DecisionTree: {'model__min_samples_spl

ValueError: 
All the 30 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\admin\miniconda3\envs\isye_env\Lib\site-packages\xgboost\data.py", line 407, in pandas_feature_info
    new_feature_types.append(_pandas_dtype_mapper[dtype.name])
                             ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'object'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\admin\miniconda3\envs\isye_env\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\admin\miniconda3\envs\isye_en